##### Imports

In [41]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from econml.dml import CausalForestDML

from gg570_d200.auxiliary_functions.forest_riesz_funcs import call_forestriesz, call_forestriesz_cross, calculate_p_value
from gg570_d200.auxiliary_functions.ate_estimation_funcs import forest_riesz_gate, forest_riesz_gate_cross, causal_dml_gate

In [42]:
root = Path.cwd().parent
data_path = root / "data"

In [43]:
np.random.seed(21)

In [44]:
df_scaled = pd.read_csv(data_path / "df_scaled.csv")

In [45]:
saved_joblib = joblib.load(data_path / "covariate_scaler.joblib")
scaler = saved_joblib['scaler']

In [46]:
df = pd.read_csv(data_path / "df.csv")

In [47]:
treatment_col = 'Training (last 3 months)'
outcome_col = 'Underemployment hours'
covariate_cols = [col for col in df_scaled.columns if col not in treatment_col and col not in outcome_col and col not in ['prop_scores']]

In [48]:
relevant_gates = set()

In [49]:
def add_relevant_gates(est):
    global relevant_gates
    if isinstance(est, list):
        for est_i in est:
            heterogeneity_importance = est_i.feature_importances_
            top_3_ids = np.argsort(heterogeneity_importance)[-3:][::-1]
            for id in top_3_ids:
                relevant_gates.add(id)
    else:
        heterogeneity_importance = est.feature_importances_
        top_3_ids = np.argsort(heterogeneity_importance)[-3:][::-1]
        for id in top_3_ids:
            relevant_gates.add(id)

---

##### Key specification: ForestRiesz (cross-validated, scaled)

In [50]:
cross_scale, est_cs = call_forestriesz_cross(df_scaled, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'], return_est=True)
add_relevant_gates(est_cs)
cross_scale

{'dr': {'est': -0.16932964703007677,
  'low': -1.811734160699806,
  'high': 1.4730748666396527,
  'p_val': 0.839862319686548},
 'plugin': {'est': -0.03742179387258689,
  'low': -0.5965905370826727,
  'high': 0.5217469493374989,
  'p_val': 0.895641953248558}}

In [51]:
add_relevant_gates(est_cs)

In [52]:
mask = df_scaled['FTPTWK'] > 0.0
print(forest_riesz_gate_cross(df_scaled, covariate_cols, treatment_col, outcome_col, est_cs, mask))

[-0.929254739721654, -1.7511477469140306, -0.10736173252927739, 0.026692258903210808]


---

##### Alternative specifications: ForestRiesz (not cross-validated, scaled)

In [53]:
no_cross_scale, est_ncs = call_forestriesz(df_scaled, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'], return_est=True)
add_relevant_gates(est_ncs)
no_cross_scale

{'dr': {'est': -0.266509157703172,
  'low': -0.8744295027331257,
  'high': 0.3414111873267817,
  'p_val': 0.3902091413738078},
 'plugin': {'est': -0.03658136219478732,
  'low': -0.2950668292160727,
  'high': 0.22190410482649808,
  'p_val': 0.7814899601055587}}

In [54]:
mask = df_scaled[f'LKWFWM_bin_(12.2, 15.0]'] > 0.0
print(forest_riesz_gate(df_scaled, covariate_cols, treatment_col, outcome_col, est_ncs, mask))
print(forest_riesz_gate(df_scaled, covariate_cols, treatment_col, outcome_col, est_ncs, ~mask))

[-0.07501008137013174, -0.722788368957177, 0.5727682062169135, 0.8204580896354163]
[-1.0231182534111636, -2.6110441894568113, 0.5648076826344841, 0.20665154423364607]


---

##### Alternative specifications: ForestRiesz (not cross-validated, not scaled)

In [55]:
no_cross_no_scale, est_ncns = call_forestriesz(df, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'], return_est=True)
add_relevant_gates(est_ncns)
no_cross_no_scale

{'dr': {'est': -0.26791976018597574,
  'low': -0.8760499672492762,
  'high': 0.3402104468773247,
  'p_val': 0.387869264561584},
 'plugin': {'est': -0.03657351864532722,
  'low': -0.2951955071691021,
  'high': 0.22204846987844767,
  'p_val': 0.7816480220116433}}

In [56]:
mask = df[f'LKWFWM_bin_(12.2, 15.0]'] > 0.0
print(forest_riesz_gate(df, covariate_cols, treatment_col, outcome_col, est_ncns, mask))
print(forest_riesz_gate(df, covariate_cols, treatment_col, outcome_col, est_ncns, ~mask))

[-0.07671833859622504, -0.7246215516470011, 0.5711848744545511, 0.816476114773995]
[-1.0233528278395965, -2.6124673635286113, 0.5657617078494184, 0.20688717012825641]


In [58]:
#print(forest_riesz_gate(df, covariate_cols, treatment_col, outcome_col, est_ncns, mask))
#mask = df[f'FTPTWK_bin_(1.5, 2.0]'] > 0.0
#print(forest_riesz_gate(df, covariate_cols, treatment_col, outcome_col, est_ncns, ~mask))

---

##### Alternative specifications: ForestRiesz (cross-validated, not scaled)

In [59]:
cross_no_scale, est_cns = call_forestriesz_cross(df, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'], return_est=True)
add_relevant_gates(est_cns)
cross_no_scale

{'dr': {'est': -0.16930556883926606,
  'low': -1.8066743620193373,
  'high': 1.4680632243408054,
  'p_val': 0.8393990435692609},
 'plugin': {'est': -0.038289273524296126,
  'low': -0.5974631242549585,
  'high': 0.5208845772063663,
  'p_val': 0.8932381100393474}}

---

##### Alternative specifications: CausalForestDML (cross-validated, scaled)

In [60]:
causal_dml = CausalForestDML(
    discrete_treatment=True,
    criterion='mse',
    n_estimators=100,
    min_samples_leaf=2,
    min_var_fraction_leaf=0.001,
    min_var_leaf_on_val=True,
    min_impurity_decrease=0.01,
    #max_samples=.8,
    max_depth=None,
    inference=True, #inference=False
    subforest_size=2, #subforest_size=1
    honest=True,
    verbose=0,
    n_jobs=-2,
    random_state=21,
    cv=3
)

In [61]:
causal_dml.fit(df_scaled[outcome_col], df_scaled[treatment_col], X=df_scaled[covariate_cols])
add_relevant_gates(causal_dml)
ate, low, high = causal_dml.ate_[0], (causal_dml.ate_-1.96*causal_dml.ate_stderr_)[0], (causal_dml.ate_+1.96*causal_dml.ate_stderr_)[0]
ate, low, high, calculate_p_value(ate, low, high)

---

In [ ]:
for i in relevant_gates:
    print(covariate_cols[i], "\t", df_scaled[covariate_cols[i]].nunique())

LKWFWM_bin_(6.6, 9.4] ,  2
FTPTWK ,  2
AGE ,  48
WRKING ,  2
REFWKD ,  31
INECAC05 ,  2
LKWFWM_bin_(12.2, 15.0] ,  2


In [ ]:
for i in relevant_gates:
    if df_scaled[covariate_cols[i]].nunique() == 2:
        mask = lambda df, i: df[covariate_cols[i]]==max(df[covariate_cols[i]]) if mask_status else df[covariate_cols[i]] < max(df[covariate_cols[i]])

        for status in [True, False]:
            mask_status = status
            print("GATE\t", covariate_cols[i], mask_status)

            print("CS\t", forest_riesz_gate_cross(df_scaled, covariate_cols, treatment_col, outcome_col, est_cs, mask(df_scaled, i)))
            print("NCS\t", forest_riesz_gate(df_scaled, covariate_cols, treatment_col, outcome_col, est_ncs, mask(df_scaled, i)))
            print("NCNS\t", forest_riesz_gate(df, covariate_cols, treatment_col, outcome_col, est_ncns, mask(df, i)))
            print("CNS\t", forest_riesz_gate_cross(df, covariate_cols, treatment_col, outcome_col, est_cns, mask(df, i)))
            print("CDML\t", causal_dml_gate(df_scaled, covariate_cols, causal_dml, mask(df_scaled, i)))
            print()

GATE	 LKWFWM_bin_(6.6, 9.4] True
CS	 [-1.3444397149341638, -3.6619397793229918, 0.9730603494546641, 0.2555282117901767]
NCS	 [-1.4224419200199938, -3.6578398141729425, 0.8129559741329546, 0.21233291691221323]
NCNS	 [-1.4181443330476964, -3.6554530237979557, 0.8191643577025629, 0.21410887820864222]
CNS	 [-1.3503598274553454, -3.6669298915106276, 0.9662102365999365, 0.2532506701559112]
CDML	 [-0.516345658408554, -5.672532884545116, 4.639841567728007, 0.8443967120275604]

GATE	 LKWFWM_bin_(6.6, 9.4] False
CS	 [-0.10756045845122579, -0.7719649116218977, 0.5568439947194462, 0.7510170145536161]
NCS	 [-0.11725232813627037, -0.7406404633643453, 0.5061358070918045, 0.7123906497124457]
NCNS	 [-0.11939998582540633, -0.742939620361153, 0.5041396487103403, 0.7074311084903859]
CNS	 [-0.1069101943962657, -0.7716852015511375, 0.5578648127586061, 0.752606324155586]
CDML	 [-0.07371371116548736, -4.784550389353275, 4.6371229670223, 0.9755335869141166]

GATE	 FTPTWK True
CS	 [-0.9292547397216547, -1.75114

---

In [49]:
aage_bins = ["(1.99, 4.0]", "(10.0, 12.0]", "(4.0, 6.0]", "(6.0, 8.0]", "(8.0, 10.0]"]

In [ ]:
for bin in aage_bins:
    mask = df_scaled[f'AAGE_bin_{bin}'] > 0.0
    print(bin)
    print(causal_dml_gate(df_scaled, covariate_cols, causal_dml, mask))
    print(causal_dml_gate(df_scaled, covariate_cols, causal_dml, ~mask),"\n")

[0.4439220481437849, -4.238005028477581, 5.12584912476515, 0.8525732674546236]
[0.43759330543594094, -5.0355959290684105, 5.910782539940292, 0.8754786424206622] 

[0.897982680518652, -7.889935927161423, 9.685901288198725, 0.8412643287892894]
[0.35747163564126666, -4.230781133585723, 4.9457244048682565, 0.8786339440360031] 

[0.2679351397235288, -4.185432863851554, 4.721303143298612, 0.9061306779659224]
[0.48565221593514063, -5.180829157094368, 6.152133588964648, 0.8665982758016404] 

[0.3244058040060629, -4.18400889770629, 4.8328205057184155, 0.8878458979772033]
[0.48432692206265154, -5.271650660785073, 6.240304504910374, 0.8690086594137885] 

[0.44230010773247985, -4.308591924174136, 5.193192139639096, 0.8552143890995381]
[0.4363769277624678, -5.223824119385391, 6.096577974910326, 0.8798931137315333] 



In [ ]:
df_scaled['REFWKD_bins'] = pd.qcut(df_scaled['REFWKD'], q=5, duplicates='drop')
refwkd_bins = df_scaled['REFWKD_bins'].cat.categories

for bin_label in refwkd_bins:
    mask = df_scaled['REFWKD_bins'] == bin_label
    print(bin_label)
    print(causal_dml_gate(df_scaled, covariate_cols, causal_dml, mask))
    print(causal_dml_gate(df_scaled, covariate_cols, causal_dml, ~mask),"\n")

(-1.64, -1.082]
[0.42587022359288473, -4.710686186352758, 5.562426633538526, 0.8709121461022447]
[0.44125835210343967, -5.057361553209365, 5.939878257416241, 0.8750202347656462] 

(-1.082, -0.303]
[0.4113043899379888, -4.9971892751234686, 5.819798054999445, 0.8815133316054495]
[0.44549428906898647, -4.983193147606602, 5.874181725744573, 0.8722190318657961] 

(-0.303, 0.365]
[0.4176196440038939, -5.343680253801295, 6.178919541809082, 0.8870231260110737]
[0.4422282745986287, -4.909741136422178, 5.794197685619434, 0.8713452215571271] 

(0.365, 1.033]
[0.4691867034825747, -5.104514015884092, 6.042887422849241, 0.8689540900194856]
[0.4296571287610749, -4.953860123733794, 5.813174381255943, 0.8756986457487133] 

(1.033, 1.701]
[0.46630647411051473, -4.80099047008072, 5.733603418301748, 0.8622482998211343]
[0.4314785952562868, -5.028410295968789, 5.891367486481362, 0.8769079916568943] 



In [60]:
df_scaled['HIQUAL22_bins'] = pd.qcut(df_scaled['HIQUAL22'], q=5, duplicates='drop')
hiqual22_bins = df_scaled['HIQUAL22_bins'].cat.categories

for bin_label in hiqual22_bins:
    mask = df_scaled['HIQUAL22_bins'] == bin_label
    print(bin_label)
    print(causal_dml_gate(df_scaled, covariate_cols, causal_dml, mask))
    print(causal_dml_gate(df_scaled, covariate_cols, causal_dml, ~mask),"\n")

(-1.1769999999999998, -0.848]
[-0.31858824545105846, -4.996984750210298, 4.3598082593081795, 0.8938223806135186]
[0.06500036056709617, -4.598281828355337, 4.7282825494895295, 0.9782049333511509] 

(-0.848, -0.68]
[-0.2605538303047884, -4.589410382080641, 4.068302721471063, 0.9060912607272438]
[-0.0814713305308577, -4.7547250831602215, 4.591782422098506, 0.9727423069762695] 

(-0.68, 0.04]
[-0.02207988373056389, -4.623927323047323, 4.579767555586194, 0.9924968035616082]
[-0.10000586058149799, -4.786816302107778, 4.586804580944781, 0.9666412604693542] 

(0.04, 1.256]
[0.038510615253818, -4.324986024408915, 4.40200725491655, 0.9861989443838624]
[-0.13971363960494965, -4.942496706803025, 4.663069427593125, 0.9545327445472578] 

(1.256, 2.191]
[0.46626520721410236, -5.446103011674126, 6.37863342610233, 0.8771618880505727]
[-0.12907502254965303, -4.680474973293997, 4.42232492819469, 0.9556736694271242] 



In [70]:
mask = df_scaled[f'FTPTWK_bin_(1.5, 2.0]'] > 0.0
print(causal_dml_gate(df_scaled, covariate_cols, causal_dml, mask))
print(causal_dml_gate(df_scaled, covariate_cols, causal_dml, ~mask))

[-0.1578066923088366, -4.897030370299807, 4.581416985682132, 0.9479646888394797]
[-0.0016853489332691828, -4.592055189909651, 4.588684492043112, 0.9994258435236247]


In [ ]:
target = "FTPTWK_bin_(1.5, 2.0]"

high_corr = (
    df_scaled.corr(numeric_only=True)[target]
    .drop(target)
    .sort_values(key=np.abs, ascending=False)
)
high_corr[high_corr.abs() >= 0.5]

---

In [17]:
max_col_mean = scaler.mean_[max_id]
max_col_std = scaler.scale_[max_id]

df_scaled[f"{max_col}_original"] = df_scaled[max_col] * max_col_std + max_col_mean

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


In [19]:
df_scaled

,"AAGE_bin_(1.99, 4.0]","AAGE_bin_(10.0, 12.0]","AAGE_bin_(4.0, 6.0]","AAGE_bin_(6.0, 8.0]","AAGE_bin_(8.0, 10.0]",AGE,"ANXIOUS_bin_(-0.01, 2.0]","ANXIOUS_bin_(2.0, 4.0]","ANXIOUS_bin_(4.0, 6.0]","ANXIOUS_bin_(6.0, 8.0]",...,"edageband_bin_(10.916, 27.8]","edageband_bin_(27.8, 44.6]","edageband_bin_(44.6, 61.4]","edageband_bin_(61.4, 78.2]","edageband_bin_(78.2, 95.0]",Training (last 3 months),Underemployment hours,prop_scores,AGE_original,AGE_original_bin
0,-0.266552,-0.418452,-0.529246,1.566203,-0.617794,0.197454,-0.884804,-0.487591,2.034686,-0.399483,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,1,10.0,0.305241,44.0,"(42.0, 45.0]"
1,-0.266552,-0.418452,-0.529246,1.566203,-0.617794,-0.247202,1.130194,-0.487591,-0.491476,-0.399483,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,0,5.0,0.550512,39.0,"(38.0, 42.0]"
2,-0.266552,-0.418452,-0.529246,-0.638487,1.618664,0.997835,1.130194,-0.487591,-0.491476,-0.399483,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,0,3.0,0.211601,53.0,"(49.0, 53.0]"
3,-0.266552,-0.418452,1.889480,-0.638487,-0.617794,-1.136514,-0.884804,-0.487591,2.034686,-0.399483,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,0,5.0,0.268558,29.0,"(27.0, 31.0]"
4,-0.266552,-0.418452,1.889480,-0.638487,-0.617794,-1.225446,1.130194,-0.487591,-0.491476,-0.399483,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,0,8.0,0.179623,28.0,"(27.0, 31.0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015,-0.266552,-0.418452,-0.529246,-0.638487,1.618664,0.908903,-0.884804,-0.487591,-0.491476,2.503235,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,1,7.0,0.455386,52.0,"(49.0, 53.0]"
2016,-0.266552,-0.418452,1.889480,-0.638487,-0.617794,-1.403308,-0.884804,-0.487591,2.034686,-0.399483,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,0,2.0,0.112617,26.0,"(16.999, 27.0]"
2017,-0.266552,2.389762,-0.529246,-0.638487,-0.617794,1.442491,1.130194,-0.487591,-0.491476,-0.399483,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,0,8.0,0.036216,58.0,"(57.0, 64.0]"
2018,-0.266552,-0.418452,-0.529246,-0.638487,1.618664,0.553179,1.130194,-0.487591,-0.491476,-0.399483,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,1,20.0,0.425192,48.0,"(45.0, 49.0]"


In [21]:
ate = causal_dml.ate(X=X_group)
lb, ub = causal_dml.ate_interval(X=X_group, alpha=0.05)

ate, (lb, ub)